# SAFE rollouts：Google Drive → Colab → 服务器（加速版 V2）

文件先下载到 Colab 临时磁盘，再通过原生 `rsync` 上传到服务器；不会经过本地电脑。每完成一个文件便删除其 Colab 临时副本。Google Drive 下载和服务器上传均支持断点续传。

In [ ]:
!apt-get -qq update && apt-get -qq install -y sshpass rsync >/dev/null
!pip -q install 'gdown>=5.2.0' 'paramiko>=3.5.0'

In [ ]:
import base64
import getpass
import hashlib
import os
from pathlib import Path
import shlex
import socket
import stat
import subprocess

import gdown
import paramiko

HOST = 'connect.bjb2.seetacloud.com'
PORT = 20559
USER = 'root'
REMOTE_DIR = '/root/autodl-tmp/datasets/safe-rollouts'
EXPECTED_HOST_KEY = 'SHA256:liZ36vNCsNcNdXeWs4f+g5ZIhPM/ZihP834vxs8Ulqc'
HOST_PUBLIC_KEY = 'ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIK3kwiFvUVgx1wVKa/LqYm6JNHZqht3d+OSpVJVG/AWa'
FILES = [
    ('13z_cdwnaJota2iHkZbhYgVALujZwtM3b', 'pi0fast_droid_0510_all.zip'),
    ('1EwaccasZjnlM9L6SEYyWqTd7d6-BR9zp', 'openvla_widowx.zip'),
]
LOCAL_DIR = Path('/content/safe-rollouts')
KNOWN_HOSTS = Path('/content/safe_known_hosts')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
KNOWN_HOSTS.write_text(
    f'[{HOST}]:{PORT} {HOST_PUBLIC_KEY}\n', encoding='utf-8'
)
password = getpass.getpass('服务器密码（输入不会显示）：')

def open_sftp():
    sock = socket.create_connection((HOST, PORT), timeout=30)
    transport = paramiko.Transport(sock)
    transport.start_client(timeout=30)
    server_key = transport.get_remote_server_key()
    fingerprint = 'SHA256:' + base64.b64encode(
        hashlib.sha256(server_key.asbytes()).digest()
    ).decode().rstrip('=')
    if fingerprint != EXPECTED_HOST_KEY:
        transport.close()
        raise RuntimeError(f'服务器指纹不匹配：{fingerprint}')
    transport.auth_password(USER, password)
    return transport, paramiko.SFTPClient.from_transport(transport)

def mkdir_p(sftp, path):
    current = ''
    for part in path.strip('/').split('/'):
        current += '/' + part
        try:
            mode = sftp.stat(current).st_mode
            if not stat.S_ISDIR(mode):
                raise RuntimeError(f'远程路径不是目录：{current}')
        except FileNotFoundError:
            sftp.mkdir(current)

def final_exists(filename):
    transport, sftp = open_sftp()
    try:
        mkdir_p(sftp, REMOTE_DIR)
        try:
            return sftp.stat(f'{REMOTE_DIR}/{filename}').st_size
        except FileNotFoundError:
            return None
    finally:
        sftp.close()
        transport.close()

ssh_command = (
    f'ssh -p {PORT} -o StrictHostKeyChecking=yes '
    f'-o UserKnownHostsFile={shlex.quote(str(KNOWN_HOSTS))} '
    '-o Compression=no -o ServerAliveInterval=30 -o ServerAliveCountMax=10 '
    '-c aes128-gcm@openssh.com'
)
child_env = os.environ.copy()
child_env['SSHPASS'] = password

for file_id, filename in FILES:
    existing_size = final_exists(filename)
    if existing_size is not None:
        print(f'服务器已存在，跳过：{filename} ({existing_size:,} bytes)')
        continue

    local_path = LOCAL_DIR / filename
    print(f'① Google Drive → Colab：{filename}')
    result = gdown.download(
        id=file_id, output=str(local_path), quiet=False,
        use_cookies=True, resume=True,
    )
    if result is None or not local_path.is_file():
        raise RuntimeError(f'Google Drive 下载失败：{filename}')
    local_size = local_path.stat().st_size
    print(f'Colab 临时文件：{local_size:,} bytes')

    remote_part = f'{REMOTE_DIR}/{filename}.part'
    print(f'② Colab → 服务器（rsync 断点续传）：{filename}')
    subprocess.run(
        [
            'sshpass', '-e', 'rsync', '-ah', '--no-compress',
            '--partial', '--append-verify', '--info=progress2',
            '-e', ssh_command, str(local_path),
            f'{USER}@{HOST}:{remote_part}',
        ],
        check=True, env=child_env,
    )

    transport, sftp = open_sftp()
    try:
        remote_size = sftp.stat(remote_part).st_size
        if remote_size != local_size:
            raise RuntimeError(
                f'大小不一致：本地 {local_size:,}，服务器 {remote_size:,}'
            )
        sftp.rename(remote_part, f'{REMOTE_DIR}/{filename}')
    finally:
        sftp.close()
        transport.close()
    local_path.unlink()
    print(f'完成并清理 Colab 临时文件：{filename} ({local_size:,} bytes)')

password = None
child_env.pop('SSHPASS', None)
print('两份数据传输结束。请断开并删除当前 Colab 运行时。')